In [1]:
!pip install python-chess pandas tqdm


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# This notebook is for generating datasets to train the CNN

import csv
import math
import random
import pandas as pd
from tqdm import tqdm

import chess
import chess.engine

random.seed(42)

In [3]:
num_positions = int(1e5)
search_depth = 12
max_moves = 80
engine_path = "/Users/adityapatil/Downloads/stockfish/stockfish-macos-m1-apple-silicon"
engine = chess.engine.SimpleEngine.popen_uci(engine_path)
output_csv = "dataset.csv"

In [4]:
def generate_random_pos(top_k=3,random_move_prob=0.3,factor=1):
    board = chess.Board()
    r = random.random()

    # More middle game and end game samples instead of start since limited games possible in start 
    if r < 0.25:
        n_moves = random.randint(1,12)      
    elif r < 0.80:
        n_moves = random.randint(13,50)     
    else:
        n_moves = random.randint(51,100) 
   
    for _ in range(n_moves):
        if board.is_game_over():
            break
        moves = list(board.legal_moves)
        
        if random.random() < random_move_prob:
            move = random.choice(moves)
            board.push(move)
            continue
        
        multipv = min(top_k, len(moves))
        info = engine.analyse(board, chess.engine.Limit(depth=5), multipv=multipv) 
        
        candidate_moves = [entry["pv"][0] for entry in info]
        scores = [entry["score"].white().score(mate_score=10000) for entry in info]
        
        if not board.turn:
            scores = [-score for score in scores]
        
        max_score = max(scores)
        exp_scores = [math.exp((score - max_score) / (100 * factor)) for score in scores]
        total = sum(exp_scores)
        probs = [e / total for e in exp_scores]
        
        move = random.choices(candidate_moves, weights=probs, k=1)[0]
        board.push(move)
    
    return board

In [5]:
def evaluate(board):
    info = engine.analyse(board,chess.engine.Limit(depth=search_depth))
    score = info["score"].white()
    
    centipawn = score.score(mate_score=10000) # To convert "mate in 5" type scores to integer
    centipawn = max(-1000, min(1000, centipawn))  
    best_move = info["pv"][0].uci() if "pv" in info else None
    
    return centipawn,best_move

In [6]:
seen = set()
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["fen", "evaluation","best_move"])
    pbar = tqdm(total=num_positions)
    while len(seen) < num_positions:
        board = generate_random_pos()
        key = " ".join(board.fen().split(" ")[:4])
        if key in seen:
            continue
        seen.add(key)
        score, best_move = evaluate(board)
        writer.writerow([board.fen(), score,best_move])
        pbar.update(1)
        if len(seen) % 10000 == 0:
            f.flush()
    pbar.close()
engine.quit()
print(f"Saved {len(seen)} unique positions into dataset !")

100%|██████████| 100000/100000 [59:25<00:00, 28.04it/s] 

Saved 100000 unique positions into dataset !
